
# Gold Zone
 

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS workspace.gold;

## 1. Fact ventas final

Crear la tabla delta **fact_ventas_final**:

a) Particionada por el campo *mes*.

b) Incluir el campo *monto_total* resolviendo el cálculo entre la cantidad del producto vendido y su precio unitario.


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.fact_ventas_final USING DELTA
PARTITIONED BY (mes) AS
SELECT
    f.id_producto,
    f.id_vendedor,
    f.cantidad,
    (f.cantidad * p.precio_unitario) AS monto_total,
    f.dia,
    f.mes,
    f.ano

FROM workspace.silver.fact_venta AS f

INNER JOIN workspace.silver.dim_producto as p
ON f.id_producto = p.id_producto;


##2. Corrección de datos

a) Truncar la partición de diciembre.

In [0]:
%sql
DELETE FROM workspace.gold.fact_ventas_final
WHERE mes = 12;


b) Actualizar la tabla descontando el 10% a la columna *monto_total* para el mes de junio.


In [0]:
%sql
UPDATE workspace.gold.fact_ventas_final
SET monto_total = monto_total * 0.90
WHERE mes = 6;

In [0]:
%sql
select * from workspace.gold.fact_ventas_final